Random Forest

In [1]:
#USED K-BEST ANOVA - 25 FEATURES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score


# Load the dataset
df = pd.read_csv(r'C:\Users\arell\Documents\1_ALF\data\malicious_2021.csv', low_memory=False)

# Select features and target columns
features = ['domain_token_count', 'path_token_count', 'avgdomaintokenlen', 'longdomaintokenlen', 'ldl_domain', 'ldl_path', 'subDirLen', 'pathurlRatio', 'argDomanRatio', 'domainUrlRatio', 'NumberofDotsinURL', 'CharacterContinuityRate', 'host_DigitCount', 'host_letter_count', 'Directory_LetterCount', 'Domain_LongestWordLength', 'sub-Directory_LongestWordLength', 'URLQueries_variable', 'delimeter_Domain', 'delimeter_Count', 'NumberRate_Domain', 'SymbolCount_URL', 'SymbolCount_Domain', 'Entropy_Domain', 'tld']

# Clean the dataset by removing NaNs and infinities in numeric columns only
df_cleaned = df.copy()
df_cleaned['tld'] = df_cleaned['tld'].astype(str)
df_cleaned['url_type'] = df_cleaned['url_type'].astype(str)

numeric_features = [f for f in features if f not in ['tld', 'url_type']]
df_cleaned = df_cleaned[np.isfinite(df_cleaned[numeric_features]).all(axis=1)]

label_encoder_tld = LabelEncoder()
label_encoder_url_type = LabelEncoder()
df_cleaned['tld_encoded'] = label_encoder_tld.fit_transform(df_cleaned['tld'])
df_cleaned['url_type_encoded'] = label_encoder_url_type.fit_transform(df_cleaned['url_type'])
X = df_cleaned[numeric_features + ['tld_encoded']]
y = df_cleaned['binary_label'] = df_cleaned['url_type'].apply(lambda x: 0 if x == 'benign' else 1)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=df_cleaned['binary_label']
)

# Initialize and fit the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)

# Predictions
y_train_pred = rf_classifier.predict(X_train)
y_test_pred = rf_classifier.predict(X_test)

# Classification reports
print("Binary Classification Report - Training Data:")
print(classification_report(y_train, y_train_pred))
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

print("\nBinary Classification Report - Test Data:")
print(classification_report(y_test, y_test_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

# Multiclass Classification
malicious_df = df_cleaned[df_cleaned['binary_label'] == 1].copy()
X_multi = malicious_df[numeric_features + ['tld_encoded']]
y_multi = malicious_df['url_type_encoded']
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.4, random_state=42, stratify=y_multi
)
rf_multiclass_classifier = RandomForestClassifier(random_state=42)
rf_multiclass_classifier.fit(X_train_multi, y_train_multi)

# Predictions and Evaluations for Multiclass
y_train_pred_multi = rf_multiclass_classifier.predict(X_train_multi)
y_test_pred_multi = rf_multiclass_classifier.predict(X_test_multi)
print("Multiclass Classification Report (Training):")
print(classification_report(y_train_multi, y_train_pred_multi))
print("Multiclass Classification Report (Test):")
print(classification_report(y_test_multi, y_test_pred_multi))

# Accuracy Summary
train_accuracy_bin = accuracy_score(y_train, y_train_pred)
test_accuracy_bin = accuracy_score(y_test, y_test_pred)
train_accuracy_multi = accuracy_score(y_train_multi, y_train_pred_multi)
test_accuracy_multi = accuracy_score(y_test_multi, y_test_pred_multi)

print(f"Binary Classification - Train Accuracy: {train_accuracy_bin:.4f}")
print(f"Binary Classification - Test Accuracy: {test_accuracy_bin:.4f}")
print(f"Multiclass Classification - Train Accuracy: {train_accuracy_multi:.4f}")
print(f"Multiclass Classification - Test Accuracy: {test_accuracy_multi:.4f}")


Binary Classification Report - Training Data:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    299672
           1       0.99      0.99      0.99    156161

    accuracy                           0.99    455833
   macro avg       0.99      0.99      0.99    455833
weighted avg       0.99      0.99      0.99    455833

Training Accuracy: 0.992402919490252

Binary Classification Report - Test Data:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98    128431
           1       0.97      0.95      0.96     66927

    accuracy                           0.97    195358
   macro avg       0.97      0.97      0.97    195358
weighted avg       0.97      0.97      0.97    195358

Test Accuracy: 0.9722458256124653
Multiclass Classification Report (Training):
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     57874
           2       1.00      0.9

Random Forest with Hyperparameter

In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.stats import randint

# Load the dataset
df = pd.read_csv(r'C:\Users\arell\Documents\1_ALF\data\malicious_2021.csv', low_memory=False)

# Select features and target columns
features = ['domain_token_count', 'path_token_count', 'avgdomaintokenlen', 'longdomaintokenlen', 'ldl_domain', 'ldl_path', 'subDirLen', 'pathurlRatio', 'argDomanRatio', 'domainUrlRatio', 'NumberofDotsinURL', 'CharacterContinuityRate', 'host_DigitCount', 'host_letter_count', 'Directory_LetterCount', 'Domain_LongestWordLength', 'sub-Directory_LongestWordLength', 'URLQueries_variable', 'delimeter_Domain', 'delimeter_Count', 'NumberRate_Domain', 'SymbolCount_URL', 'SymbolCount_Domain', 'Entropy_Domain', 'tld']

# Clean the dataset by removing NaNs and infinities in numeric columns only
df_cleaned = df.copy()
df_cleaned['tld'] = df_cleaned['tld'].astype(str)
df_cleaned['url_type'] = df_cleaned['url_type'].astype(str)

numeric_features = [f for f in features if f not in ['tld', 'url_type']]
df_cleaned = df_cleaned[np.isfinite(df_cleaned[numeric_features]).all(axis=1)]

label_encoder_tld = LabelEncoder()
label_encoder_url_type = LabelEncoder()
df_cleaned['tld_encoded'] = label_encoder_tld.fit_transform(df_cleaned['tld'])
df_cleaned['url_type_encoded'] = label_encoder_url_type.fit_transform(df_cleaned['url_type'])

X = df_cleaned[numeric_features + ['tld_encoded']]
y = df_cleaned['binary_label'] = df_cleaned['url_type'].apply(lambda x: 0 if x == 'benign' else 1)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=df_cleaned['binary_label']
)

# Initialize and fit the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)
# Hyperparameter tuning using Grid Search
random_params = {
    'n_estimators': randint(100, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5)
}

rf_classifier = RandomizedSearchCV(estimator=rf_classifier, param_distributions=random_params, n_iter=50, cv=3, n_jobs=-1, random_state=42)
rf_classifier.fit(X_train, y_train)

# Predictions
y_train_pred = rf_classifier.predict(X_train)
y_test_pred = rf_classifier.predict(X_test)

# Classification reports
print("Binary Classification Report - Training Data:")
print(classification_report(y_train, y_train_pred))
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

print("\nBinary Classification Report - Test Data:")
print(classification_report(y_test, y_test_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

# Multiclass Classification
malicious_df = df_cleaned[df_cleaned['binary_label'] == 1].copy()
X_multi = malicious_df[numeric_features + ['tld_encoded']]
y_multi = malicious_df['url_type_encoded']
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42, stratify=y_multi
)
rf_multiclass_classifier = RandomForestClassifier(random_state=42)
rf_multiclass_classifier = RandomizedSearchCV(estimator=rf_multiclass_classifier, param_distributions=random_params, n_iter=50, cv=3, n_jobs=-1, random_state=42)
rf_multiclass_classifier.fit(X_train_multi, y_train_multi)

# Predictions and Evaluations for Multiclass
y_train_pred_multi = rf_multiclass_classifier.predict(X_train_multi)
y_test_pred_multi = rf_multiclass_classifier.predict(X_test_multi)
print("Multiclass Classification Report (Training):")
print(classification_report(y_train_multi, y_train_pred_multi))
print("Multiclass Classification Report (Test):")
print(classification_report(y_test_multi, y_test_pred_multi))

# Accuracy Summary
train_accuracy_bin = accuracy_score(y_train, y_train_pred)
test_accuracy_bin = accuracy_score(y_test, y_test_pred)
train_accuracy_multi = accuracy_score(y_train_multi, y_train_pred_multi)
test_accuracy_multi = accuracy_score(y_test_multi, y_test_pred_multi)

print(f"Binary Classification - Train Accuracy: {train_accuracy_bin:.4f}")
print(f"Binary Classification - Test Accuracy: {test_accuracy_bin:.4f}")
print(f"Multiclass Classification - Train Accuracy: {train_accuracy_multi:.4f}")
print(f"Multiclass Classification - Test Accuracy: {test_accuracy_multi:.4f}")

Binary Classification Report - Training Data:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    299672
           1       0.99      0.98      0.98    156161

    accuracy                           0.99    455833
   macro avg       0.99      0.99      0.99    455833
weighted avg       0.99      0.99      0.99    455833

Training Accuracy: 0.9890683649494442

Binary Classification Report - Test Data:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98    128431
           1       0.97      0.95      0.96     66927

    accuracy                           0.97    195358
   macro avg       0.97      0.97      0.97    195358
weighted avg       0.97      0.97      0.97    195358

Test Accuracy: 0.9727269935195897
Multiclass Classification Report (Training):
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     67520
           2       1.00      0.